# 🏛️ Sistema Multiagente RAG — Orden Público Cauca 2026
**Universidad del Cauca · Facultad de Ingeniería Electrónica y Telecomunicaciones**  
Curso: *Aplicaciones de la IA Generativa* · Metodología CRISP-DM

---

## Arquitectura
```
Consulta (lenguaje natural)
        │
        ▼
┌──────────────────┐
│ Agente 1         │  text-embedding-3-large + ChromaDB
│ Recuperador RAG  │  → top-5 documentos relevantes
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Agente 2         │  GPT-4.1 + few-shot prompting
│ Generador        │  → reporte institucional
└────────┬─────────┘
         │
         ▼
┌──────────────────┐
│ Agente 3         │  GPT-4o-mini
│ Verificador      │  → Faithfulness Score + reporte corregido
└──────────────────┘
```

---

## 0 · Instalación de dependencias
Ejecutar una sola vez por sesión de Colab.

In [ ]:
!pip install -q openai chromadb faiss-cpu pandas openpyxl numpy \
                python-dateutil thefuzz python-Levenshtein \
                python-dotenv tqdm pydantic

## 1 · Configuración de credenciales

En Colab, ve a **Secrets** (ícono de llave 🔑 en el panel izquierdo) y agrega:
- `OPENAI_API_KEY` → tu clave de OpenAI

Si estás en local, crea un archivo `.env` en la raíz del proyecto con:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
import os

# ── Opción A: Google Colab Secrets ────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("[OK] API key cargada desde Colab Secrets.")
except Exception:
    # ── Opción B: .env local ──────────────────────────────────────────────────
    from dotenv import load_dotenv
    load_dotenv()
    if os.environ.get("OPENAI_API_KEY"):
        print("[OK] API key cargada desde .env")
    else:
        print("[WARN] No se encontró OPENAI_API_KEY. Configúrala antes de continuar.")

# Verificación (no imprime la clave completa)
key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key detectada: {'sk-...' + key[-4:] if key else 'NO CONFIGURADA'}")

## 2 · Montar Google Drive y clonar el repositorio

Sube el archivo Excel del Cauca a tu Drive en la carpeta `ProyectoMultiagentes/data/raw/`.

In [ ]:
# ── Montar Drive (solo en Colab) ──────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/ProyectoMultiagentes"
    print(f"Drive montado. Ruta base: {DRIVE_BASE}")
except Exception:
    DRIVE_BASE = "."   # fallback local
    print("Drive no disponible. Usando directorio actual.")

import sys
sys.path.insert(0, DRIVE_BASE)

# Ruta al Excel (ajusta el nombre del archivo)
RUTA_EXCEL = os.path.join(DRIVE_BASE, "data", "raw", "informe_seguridad_cauca_2024.xlsx")
print(f"Excel esperado en: {RUTA_EXCEL}")
print(f"¿Existe?: {os.path.exists(RUTA_EXCEL)}")

## 3 · Preprocesamiento del dataset

Limpia y normaliza el Excel. El resultado se guarda en `data/processed/`.

In [ ]:
from src.preprocessing import preprocesar

df = preprocesar(RUTA_EXCEL, guardar=True)
print(f"\nDataset limpio: {len(df):,} registros")
df.head()

In [ ]:
# Estadísticas básicas del dataset
print("Distribución por categoría de hecho:")
print(df["categoria_hecho"].value_counts())
print("\nDistribución por municipio (top 10):")
print(df["municipio"].value_counts().head(10))
print("\nDistribución por semestre:")
print(df["semestre"].value_counts())

## 4 · Construcción del vector store

Genera embeddings con `text-embedding-3-large` e indexa en ChromaDB.
**Este paso consume tokens de la API de OpenAI.** Solo ejecutarlo cuando el dataset cambie.

In [ ]:
from src.vector_store import construir_vector_store, cargar_chroma

# Construir desde cero (descomentar solo si el dataset cambió)
# coleccion = construir_vector_store()

# Cargar un índice ya existente (recomendado para sesiones posteriores)
coleccion = cargar_chroma()
print(f"Vector store cargado: {coleccion.count():,} documentos indexados.")

## 5 · Inicializar el pipeline

Instancia los tres agentes y el orquestador.

In [ ]:
from src.pipeline import Pipeline

pipeline = Pipeline(coleccion=coleccion, verbose=True)
print("Pipeline inicializado y listo.")

## 6 · Consultas al sistema

Escribe tu consulta en lenguaje natural y ejecuta la celda.

In [ ]:
# ── Escribe aquí tu consulta ──────────────────────────────────────────────────
CONSULTA = "Top 3 municipios con más hechos de orden público en 2024"

resultado = pipeline.ejecutar(CONSULTA)

print("\n" + "="*60)
print("REPORTE FINAL VALIDADO")
print("="*60)
print(resultado.reporte_final)

In [ ]:
# Más consultas de ejemplo — ejecuta cada una por separado
consultas_ejemplo = [
    "¿Cuáles fueron los hechos de orden público en Toribío durante el primer semestre?",
    "Genera un informe mensual de eventos de hostigamiento para mayo de 2024",
    "¿Qué municipios registraron ataques con drones y cuáles fueron las afectaciones?",
    "Resume los eventos de desplazamiento forzado del segundo semestre",
    "¿Cuántos eventos de bloqueo vial hubo en el Cauca en 2024?",
]

for c in consultas_ejemplo:
    print(f"\n>>> {c}")

## 7 · Métricas de evaluación

Monitoreo de los indicadores clave definidos en CONTEXTO.md.

In [ ]:
# Faithfulness Score del último resultado
v = resultado.verificacion

print("─── Métricas de la última consulta ───")
print(f"Faithfulness Score : {v.faithfulness_score:.3f}  (objetivo ≥ 0.85)")
print(f"Alucinaciones      : {len(v.alucinaciones)}         (objetivo ≤ 10%)")
print(f"Documentos recup.  : {len(resultado.documentos_recuperados)}         (objetivo Recall@5 ≥ 0.80)")
print(f"Tiempo total       : {resultado.tiempo_total_seg:.1f} seg")
print(f"Estado             : {'APROBADO ✓' if v.aprobado else 'REQUIERE REVISIÓN ✗'}")

if v.alucinaciones:
    print("\nPosibles alucinaciones detectadas:")
    for a in v.alucinaciones:
        print(f"  • {a}")